# AI-Driven CPU Process Scheduling for Heterogeneous Multi-Core Processors

### A Deep Q-Network (DQN) implementation from scratch using NumPy.

This notebook simulates an Intel Raptor Lake-style Heterogeneous Multi-core Processor (HMP) and compares AI-driven scheduling against Round-Robin and Static baselines.

## 1. Setup and Reproducibility

This section includes all necessary imports and sets up reproducibility by fixing random seeds.

In [1]:
import numpy as np
import random
import collections
import json
import os
import argparse
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

## 2. Hardware Model (`HardwareConfig`)

This class defines the microarchitectural parameters for the simulated heterogeneous cores (P-cores and E-cores).

In [2]:
class HardwareConfig:
    """
    Models the key microarchitectural parameters of heterogeneous core types.
    P-cores: High IPC, Hyperthreading, high power draw.
    E-cores: Lower IPC, no HT, energy-efficient — ideal for background tasks.
    """
    P_CORE_IPC      = 5.0    # Instructions Per Cycle
    E_CORE_IPC      = 2.5
    P_CORE_POWER_W  = 15.0   # Thermal Design Power (W)
    E_CORE_POWER_W  = 3.0
    CLOCK_GHZ       = 3.2
    NUM_P_CORES     = 2
    NUM_E_CORES     = 4

## 3. Task/Process Model (`TASK_CLASSES`, `generate_process`)

Defines different classes of tasks (compute, memory, I/O bound) and a function to generate synthetic processes with their performance counter profiles.

In [3]:
TASK_CLASSES = {
    # name: (cpu_burst_ms_range, mem_intensity_base, ipc_penalty_on_e_core)
    "compute": ((20, 80), 0.10, 0.55),   # Compute-bound  → prefers P-core
    "memory":  ((10, 40), 0.70, 0.80),   # Memory-bound   → E-core OK
    "io":      (( 2, 15), 0.30, 0.90),   # IO-bound       → E-core optimal
}

def generate_process(task_class=None):
    """
    Generates a synthetic process with a hardware performance counter profile.

    Returns
    -------
    state : np.ndarray, shape (5,)
        Normalised feature vector:
        [cpu_burst_norm, mem_intensity, llc_miss_rate, branch_miss_rate, ipc_norm]
    task_class : str
        Ground-truth class label.
    cpu_burst_ms : float
        Raw CPU burst in milliseconds (used for simulation timing).
    """
    if task_class is None:
        task_class = random.choice(list(TASK_CLASSES.keys()))

    burst_range, mem_base, _ = TASK_CLASSES[task_class]
    cpu_burst_ms = np.random.uniform(*burst_range)
    mem_intensity = np.clip(mem_base + np.random.normal(0, 0.05), 0, 1)
    llc_miss_rate = np.clip(mem_intensity * 0.6 + np.random.uniform(0, 0.15), 0, 1)
    branch_misses = np.random.uniform(0.02, 0.12)
    ipc_observed  = np.clip(
        HardwareConfig.P_CORE_IPC * (1 - llc_miss_rate * 0.4)
        + np.random.normal(0, 0.1), 0.5, HardwareConfig.P_CORE_IPC
    )
    state = np.array([
        cpu_burst_ms / 80.0,                         # normalised burst
        mem_intensity,                                # memory intensity
        llc_miss_rate,                                # LLC miss rate
        branch_misses,                                # branch misprediction
        ipc_observed / HardwareConfig.P_CORE_IPC,    # normalised IPC
    ], dtype=np.float32)

    return state, task_class, cpu_burst_ms

## 4. HMP Environment (`HMPEnvironment`)

This class simulates the heterogeneous multi-core processor scheduling environment, handling task execution and calculating rewards based on execution time and energy consumption.

In [4]:
class HMPEnvironment:
    """
    Simulates a Heterogeneous Multi-Core Processor scheduling environment.

    Action Space
    ------------
    0 : Schedule process on P-core cluster
    1 : Schedule process on E-core cluster

    Reward
    ------
    R = -(w_time * exec_time_norm + w_energy * energy_norm)
    Penalises both latency (0.6 weight) and energy (0.4 weight).
    """

    W_TIME   = 0.6
    W_ENERGY = 0.4

    def __init__(self):
        self.reset_stats()

    def reset_stats(self):
        self.total_tasks = 0
        self.total_edp   = 0.0

    def reset(self):
        self.reset_stats()
        return generate_process()

    def step(self, state, action, cpu_burst_ms):
        """
        Executes the scheduling decision and returns metrics.

        Parameters
        ----------
        state : np.ndarray   — 5-dim feature vector
        action : int         — 0 = P-core, 1 = E-core
        cpu_burst_ms : float — raw burst duration

        Returns
        -------
        next_obs : tuple — (next_state, next_class, next_burst)
        reward   : float
        exec_ms  : float — execution time in milliseconds
        energy_mj: float — energy consumed in millijoules
        edp      : float — Energy-Delay Product (J·s)
        """
        task_class = self._classify(state)
        _, _, ipc_penalty_e = TASK_CLASSES[task_class]
        hw = HardwareConfig

        if action == 0:   # P-core
            effective_ipc = hw.P_CORE_IPC * (1 - state[2] * 0.3)
            power_w       = hw.P_CORE_POWER_W
        else:             # E-core
            effective_ipc = hw.E_CORE_IPC * ipc_penalty_e
            power_w       = hw.E_CORE_POWER_W

        clock_hz      = hw.CLOCK_GHZ * 1e9
        instructions  = cpu_burst_ms * 1e-3 * clock_hz
        cycles        = instructions / effective_ipc
        exec_time_s   = cycles / clock_hz
        energy_j      = power_w * exec_time_s
        edp           = energy_j * exec_time_s          # J·s

        exec_ms   = exec_time_s * 1e3
        energy_mj = energy_j * 1e3

        reward = -(self.W_TIME   * exec_ms / 80.0
                 + self.W_ENERGY * energy_mj / 500.0)

        self.total_tasks += 1
        self.total_edp   += edp

        next_obs = generate_process()
        return next_obs, reward, exec_ms, energy_mj, edp

    @staticmethod
    def _classify(state):
        """Infers task class from hardware counter features."""
        if state[0] > 0.45 and state[1] < 0.25:
            return "compute"
        elif state[1] > 0.55:
            return "memory"
        return "io"

## 5. Neural Network (`NeuralNetwork`)

This class implements a simple 3-layer fully-connected neural network with ReLU activations, including He weight initialization and manual backpropagation for SGD parameter updates. This forms the core of our DQN's function approximator.

In [5]:
class NeuralNetwork:
    """
    3-layer fully-connected network with ReLU activations.
    Architecture: Input(5) → Hidden(32) → Hidden(16) → Output(2)

    Implements:
    - He weight initialisation (optimal for ReLU)
    - Manual backpropagation via chain rule
    - SGD parameter updates
    """

    def __init__(self, layer_sizes=(5, 32, 16, 2), lr=1e-3):
        self.lr = lr
        self.weights, self.biases = [], []
        for i in range(len(layer_sizes) - 1):
            w = np.random.randn(layer_sizes[i], layer_sizes[i+1]).astype(np.float32)
            w *= np.sqrt(2.0 / layer_sizes[i])   # He init
            self.weights.append(w)
            self.biases.append(np.zeros((1, layer_sizes[i+1]), dtype=np.float32))
        self.cache = {}

    @staticmethod
    def _relu(x):  return np.maximum(0, x)

    @staticmethod
    def _relu_grad(x): return (x > 0).astype(np.float32)

    def predict(self, x):
        """Forward pass without caching (used during inference)."""
        h = x
        for i in range(len(self.weights) - 1):
            h = self._relu(h @ self.weights[i] + self.biases[i])
        return h @ self.weights[-1] + self.biases[-1]

    def forward(self, x):
        """Forward pass with activation caching (used during training)."""
        self.cache = {"a0": x}
        h = x
        for i in range(len(self.weights) - 1):
            z = h @ self.weights[i] + self.biases[i]
            h = self._relu(z)
            self.cache[f"z{i+1}"] = z
            self.cache[f"a{i+1}"] = h
        out = h @ self.weights[-1] + self.biases[-1]
        self.cache[f"a{len(self.weights)}"] = out
        return out

    def backward(self, x, y_true):
        """MSE loss backpropagation. Returns scalar loss."""
        y_pred = self.forward(x)
        loss   = float(np.mean((y_pred - y_true) ** 2))
        dout   = 2 * (y_pred - y_true) / y_pred.shape[0]

        grads_w, grads_b = [], []
        d = dout
        for i in reversed(range(len(self.weights))):
            a_prev = self.cache[f"a{i}"]
            grads_w.insert(0, a_prev.T @ d)
            grads_b.insert(0, d.sum(axis=0, keepdims=True))
            if i > 0:
                d = (d @ self.weights[i].T) * self._relu_grad(self.cache[f"z{i}"])

        for i in range(len(self.weights)):
            self.weights[i] -= self.lr * grads_w[i]
            self.biases[i]  -= self.lr * grads_b[i]
        return loss

    def copy_weights_from(self, other):
        self.weights = [w.copy() for w in other.weights]
        self.biases  = [b.copy() for b in other.biases]

## 6. DQN Agent (`DQNAgent`)

This class implements the Deep Q-Network agent, including key components like the online and target networks, a replay buffer, and an ε-greedy policy for action selection. It handles the Bellman update for training.

In [6]:
class DQNAgent:
    """
    Deep Q-Network agent for HMP scheduling.

    Key Components
    --------------
    Online Network  : Updated every training step via Bellman equation.
    Target Network  : Frozen copy of online network, updated every C steps.
                      Prevents oscillating targets during training.
    Replay Buffer   : Stores (s, a, r, s') transitions. Random sampling
                      breaks temporal correlations in experience data.
    ε-Greedy Policy : Explores randomly (ε=1.0) initially, gradually
                      shifts to exploitation (ε→0.05) as Q-values improve.

    Bellman Update
    --------------
    Q(s, a) ← r + γ · max_a' Q_target(s', a')
    """

    def __init__(self,
                 state_dim=5, action_dim=2,
                 lr=1e-3, gamma=0.95,
                 epsilon_start=1.0, epsilon_end=0.05, epsilon_decay=0.995,
                 buffer_size=5000, batch_size=64, target_update_freq=50):

        self.action_dim         = action_dim
        self.gamma              = gamma
        self.epsilon            = epsilon_start
        self.epsilon_end        = epsilon_end
        self.epsilon_decay      = epsilon_decay
        self.batch_size         = batch_size
        self.target_update_freq = target_update_freq
        self.steps              = 0
        self.losses             = []

        self.online_net = NeuralNetwork((state_dim, 32, 16, action_dim), lr)
        self.target_net = NeuralNetwork((state_dim, 32, 16, action_dim), lr)
        self.target_net.copy_weights_from(self.online_net)

        self.buffer = collections.deque(maxlen=buffer_size)

    def select_action(self, state):
        """ε-greedy action selection."""
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.action_dim)
        return int(np.argmax(self.online_net.predict(state.reshape(1, -1))))

    def store(self, s, a, r, s_next):
        self.buffer.append((s, a, float(r), s_next))

    def train_step(self):
        """Single training step: sample mini-batch, compute Bellman targets, backprop."""
        if len(self.buffer) < self.batch_size:
            return None

        batch      = random.sample(self.buffer, self.batch_size)
        states     = np.array([t[0] for t in batch], dtype=np.float32)
        actions    = np.array([t[1] for t in batch])
        rewards    = np.array([t[2] for t in batch], dtype=np.float32)
        next_states= np.array([t[3] for t in batch], dtype=np.float32)

        # Bellman targets from frozen target network
        q_next   = self.target_net.predict(next_states)
        q_target = rewards + self.gamma * np.max(q_next, axis=1)

        # Only update Q-values for the taken actions
        q_labels = self.online_net.predict(states)
        q_labels[np.arange(self.batch_size), actions] = q_target

        loss = self.online_net.backward(states, q_labels)
        self.losses.append(loss)

        # Decay exploration rate
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)

        # Periodically sync target network
        self.steps += 1
        if self.steps % self.target_update_freq == 0:
            self.target_net.copy_weights_from(self.online_net)

        return loss

## 7. Baseline Schedulers (`RoundRobinScheduler`, `StaticScheduler`)

These classes implement simple baseline scheduling policies for comparison with the DQN agent.

In [7]:
class RoundRobinScheduler:
    """
    Alternates P-core / E-core assignments regardless of task profile.
    Closest analogue to Linux CFS on an HMP system.
    """
    def __init__(self):
        self._turn = 0

    def select_action(self, state):
        action = self._turn % 2
        self._turn += 1
        return action


class StaticScheduler:
    """
    Always assigns to P-core. Models a naive 'use the fastest core' policy.
    Fast but energy-inefficient for I/O-bound workloads.
    """
    def select_action(self, state):
        return 0

## 8. Simulation Runner (`run_simulation`)

This function executes the simulation for a given scheduler and number of tasks, collecting performance metrics.

In [8]:
def run_simulation(scheduler, num_tasks=1200, is_dqn=False, verbose=True):
    """
    Run one scheduler for `num_tasks` scheduling events.

    Parameters
    ----------
    scheduler : object with .select_action(state) method
    num_tasks : int
    is_dqn    : bool — if True, calls scheduler.store() and .train_step()
    verbose   : bool — print progress

    Returns
    -------
    metrics : dict — rewards, exec_times, energies, edps, actions
    """
    env = HMPEnvironment()
    state, _, cpu_burst_ms = env.reset()

    metrics = {k: [] for k in ["rewards", "exec_times", "energies", "edps", "actions"]}

    for step in range(num_tasks):
        action = scheduler.select_action(state)
        (next_state, _, next_burst), reward, exec_ms, energy_mj, edp = \
            env.step(state, action, cpu_burst_ms)

        if is_dqn:
            scheduler.store(state, action, reward, next_state)
            scheduler.train_step()

        metrics["rewards"].append(float(reward))
        metrics["exec_times"].append(float(exec_ms))
        metrics["energies"].append(float(energy_mj))
        metrics["edps"].append(float(edp * 1e6))   # µJ·s
        metrics["actions"].append(int(action))

        state, cpu_burst_ms = next_state, next_burst

        if verbose and (step + 1) % 400 == 0:
            print(f"  Step {step+1}/{num_tasks}  "
                  f"ε={getattr(scheduler, 'epsilon', '-'):.4f}  "
                  f"Avg EDP={np.mean(metrics['edps']):.2f} µJ·s")

    return metrics

## 9. Plotting Functions

This section contains functions for visualizing the simulation results, including learning curves, EDP comparisons, and core assignment policies.

In [9]:
def _theme():
    plt.rcParams.update({
        "figure.facecolor": "#0D1B2A", "axes.facecolor": "#1A3A5C",
        "axes.edgecolor": "#00B4D8",   "axes.labelcolor": "#F0F4F8",
        "xtick.color": "#F0F4F8",      "ytick.color": "#F0F4F8",
        "text.color": "#F0F4F8",       "grid.color": "#264D73",
        "grid.linestyle": "--",         "grid.alpha": 0.5,
    })

COLORS = {"dqn": "#00B4D8", "rr": "#F77F00", "static": "#E63946",
          "green": "#2DC653", "navy": "#0D1B2A"}

def smooth(data, window=40):
    return np.convolve(data, np.ones(window)/window, mode="valid")

def plot_all(dqn_m, rr_m, st_m, dqn_agent, outdir="figures"):
    os.makedirs(outdir, exist_ok=True)
    _theme()

    # ── Fig 1: Learning Curve ──
    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor("#0D1B2A")
    x = np.arange(len(smooth(dqn_m["rewards"]))) + 20
    ax.plot(x, smooth(dqn_m["rewards"]),  color=COLORS["dqn"],    lw=2.2, label="DQN AI Scheduler")
    ax.plot(np.arange(len(smooth(rr_m["rewards"])))+20, smooth(rr_m["rewards"]),
            color=COLORS["rr"], lw=1.8, ls="--", label="Round-Robin")
    ax.plot(np.arange(len(smooth(st_m["rewards"])))+20, smooth(st_m["rewards"]),
            color=COLORS["static"], lw=1.8, ls=":", label="Static (P-core)")
    ax.axvline(x=500, color=COLORS["green"], lw=1.2, ls="--", alpha=0.7, label="Convergence (~500 tasks)")
    ax.set_title("Reward Convergence Over Training", fontsize=14, pad=10)
    ax.set_xlabel("Task / Scheduling Events"); ax.set_ylabel("Smoothed Reward (higher = better)")
    ax.legend(facecolor="#1A3A5C", edgecolor="#00B4D8", labelcolor="#F0F4F8"); ax.grid(True)
    plt.tight_layout(); plt.savefig(f"{outdir}/fig1_learning_curve.png", dpi=150, bbox_inches="tight"); plt.close()

    # ── Fig 2: EDP Comparison ──
    dqn_edp = np.mean(dqn_m["edps"]); rr_edp = np.mean(rr_m["edps"]); st_edp = np.mean(st_m["edps"])
    fig, ax = plt.subplots(figsize=(8, 5)); fig.patch.set_facecolor("#0D1B2A")
    bars = ax.bar(["DQN AI\nScheduler","Round\nRobin","Static\n(P-core)"],
                  [dqn_edp, rr_edp, st_edp],
                  color=[COLORS["dqn"],COLORS["rr"],COLORS["static"]], width=0.5, edgecolor=COLORS["navy"])
    for bar, val in zip(bars, [dqn_edp, rr_edp, st_edp]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.0002,
                f"{val:.4f}", ha="center", fontsize=11, fontweight="bold")
    reduction = 100*(rr_edp-dqn_edp)/rr_edp
    ax.annotate(f"−{reduction:.1f}% vs RR", xy=(0, dqn_edp), xytext=(0.5, dqn_edp+(rr_edp-dqn_edp)*0.5),
                arrowprops=dict(arrowstyle="->", color=COLORS["green"]),
                color=COLORS["green"], fontsize=10, fontweight="bold")
    ax.set_title("Energy-Delay Product (EDP) — Lower is Better"); ax.set_ylabel("Mean EDP (µJ·s)")
    ax.grid(True, axis="y"); plt.tight_layout()
    plt.savefig(f"{outdir}/fig2_edp_comparison.png", dpi=150, bbox_inches="tight"); plt.close()

    # ── Fig 3: Exec Time + Cumulative Energy ──
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    fig.patch.set_facecolor("#0D1B2A"); [a.set_facecolor("#1A3A5C") for a in [ax1,ax2]]
    window, n = 100, len(dqn_m["exec_times"])
    for m, c, ls, lab in [(dqn_m,COLORS["dqn"],"-","DQN"),(rr_m,COLORS["rr"],"--","RR"),(st_m,COLORS["static"],":","Static")]:
        roll = [np.mean(m["exec_times"][i:i+window]) for i in range(0, n-window, window)]
        ax1.plot(roll, color=c, lw=2, ls=ls, marker="o", ms=4, label=lab)
    ax1.set_title("Execution Time per Window (ms)"); ax1.set_xlabel("Window (100 tasks each)")
    ax1.set_ylabel("Mean Exec Time (ms)"); ax1.legend(facecolor="#1A3A5C",edgecolor="#00B4D8",labelcolor="#F0F4F8"); ax1.grid(True)
    t = np.arange(n)
    dqn_c = np.cumsum(dqn_m["energies"]); rr_c = np.cumsum(rr_m["energies"]); st_c = np.cumsum(st_m["energies"])
    ax2.plot(t,dqn_c,color=COLORS["dqn"],lw=2,label="DQN")
    ax2.plot(t,rr_c, color=COLORS["rr"], lw=1.8,ls="--",label="Round-Robin")
    ax2.plot(t,st_c, color=COLORS["static"],lw=1.8,ls=":",label="Static")
    ax2.fill_between(t,dqn_c,st_c,alpha=0.15,color=COLORS["green"],label="DQN energy saved vs Static")
    ax2.set_title("Cumulative Energy Consumption (mJ)"); ax2.set_xlabel("Tasks Processed")
    ax2.set_ylabel("Cumulative Energy (mJ)"); ax2.legend(facecolor="#1A3A5C",edgecolor="#00B4D8",labelcolor="#F0F4F8"); ax2.grid(True)
    plt.tight_layout(); plt.savefig(f"{outdir}/fig3_exec_energy.png", dpi=150, bbox_inches="tight"); plt.close()

    # ── Fig 4: Core Policy Pie Charts ──
    fig, axes = plt.subplots(1, 3, figsize=(13, 4)); fig.patch.set_facecolor("#0D1B2A")
    for ax, m, title, c in zip(axes, [dqn_m,rr_m,st_m],
                                ["DQN AI Scheduler","Round-Robin","Static (P-core)"],
                                [COLORS["dqn"],COLORS["rr"],COLORS["static"]]):
        ax.set_facecolor("#1A3A5C")
        p = m["actions"].count(0); e = len(m["actions"])-p
        _, _, auts = ax.pie([p,e], labels=["P-core","E-core"],colors=[c,"#264D73"],
                            autopct="%1.1f%%",startangle=140,
                            textprops={"color":"#F0F4F8","fontsize":11},
                            wedgeprops={"edgecolor":"#0D1B2A","linewidth":2})
        for at in auts: at.set_color("#0D1B2A"); at.set_fontweight("bold")
        ax.set_title(title, color="#F0F4F8", fontsize=11)
    plt.suptitle("Core Assignment Policy Comparison", fontsize=13, y=1.02)
    plt.tight_layout(); plt.savefig(f"{outdir}/fig4_core_policy.png", dpi=150, bbox_inches="tight"); plt.close()

    # ── Fig 5: DQN Loss Curve ──
    if dqn_agent.losses:
        fig, ax = plt.subplots(figsize=(10, 4)); fig.patch.set_facecolor("#0D1B2A")
        ls = smooth(dqn_agent.losses, window=20)
        ax.plot(ls, color=COLORS["dqn"], lw=1.8, label="MSE Training Loss")
        ax.fill_between(np.arange(len(ls)), ls, alpha=0.2, color=COLORS["dqn"])
        ax.set_title("DQN Training Loss"); ax.set_xlabel("Training Step"); ax.set_ylabel("MSE Loss")
        ax.legend(facecolor="#1A3A5C",edgecolor="#00B4D8",labelcolor="#F0F4F8"); ax.grid(True)
        plt.tight_layout(); plt.savefig(f"{outdir}/fig5_loss_curve.png", dpi=150, bbox_inches="tight"); plt.close()

    print(f"\n✅  All figures saved to: {outdir}/")
    return {"dqn_edp": dqn_edp, "rr_edp": rr_edp, "st_edp": st_edp}

## 10. Main Entry Point (`main`)

This function orchestrates the entire simulation, initializes schedulers, runs the simulations, computes summary metrics, generates plots, and optionally saves the results.

In [14]:
def main(num_tasks=1200, output_dir="figures", save_results=True):
    print("=" * 65)
    print("  HMP-DQN Scheduler Simulation")
    print(f"  Tasks: {num_tasks}  |  Seed: {SEED}")
    print("=" * 65)

    # Initialise schedulers
    dqn = DQNAgent()
    rr  = RoundRobinScheduler()
    st  = StaticScheduler()

    # Run simulations
    print("\n[1/3] Training DQN agent...")
    dqn_m = run_simulation(dqn, num_tasks, is_dqn=True)
    print(f"      Final ε = {dqn.epsilon:.4f}  |  Buffer = {len(dqn.buffer)}")

    print("[2/3] Running Round-Robin baseline...")
    rr_m  = run_simulation(rr,  num_tasks, verbose=False)

    print("[3/3] Running Static (P-core) baseline...")
    st_m  = run_simulation(st,  num_tasks, verbose=False)

    # Compute summary metrics
    def avg(m, k): return float(np.mean(m[k]))

    results = {
        "dqn_edp":           avg(dqn_m, "edps"),
        "rr_edp":            avg(rr_m,  "edps"),
        "st_edp":            avg(st_m,  "edps"),
        "edp_reduction_rr":  round(100*(avg(rr_m,"edps")-avg(dqn_m,"edps"))/avg(rr_m,"edps"),2),
        "exec_reduction_rr": round(100*(avg(rr_m,"exec_times")-avg(dqn_m,"exec_times"))/avg(rr_m,"exec_times"),2),
        "energy_vs_static":  round(100*(avg(st_m,"energies")-avg(dqn_m,"energies"))/avg(st_m,"energies"),2),
        "dqn_p_core_pct":    round(100*dqn_m["actions"].count(0)/len(dqn_m["actions"]),1),
        "epsilon_final":     round(float(dqn.epsilon),4),
    }

    print("\n" + "=" * 65)
    print("  RESULTS SUMMARY")
    print("=" * 65)
    print(f"  EDP Reduction vs Round-Robin : {results['edp_reduction_rr']:+.2f}%")
    print(f"  Exec Time Improvement        : {results['exec_reduction_rr']:+.2f}%")
    print(f"  Energy Savings vs Static     : {results['energy_vs_static']:+.2f}%")
    print(f"  DQN P-core usage             : {results['dqn_p_core_pct']}% ")
    print(f"  Final ε (exploration)        : {results['epsilon_final']}")

    # Generate figures
    plot_all(dqn_m, rr_m, st_m, dqn, output_dir)

    if save_results:
        with open("results.json", "w") as f:
            json.dump(results, f, indent=2)
        print("  Results saved → results.json")

    return results


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="HMP DQN Scheduler Simulation")
    parser.add_argument("--tasks",   type=int, default=1200, help="Number of scheduling events")
    parser.add_argument("--outdir",  type=str, default="figures", help="Output directory for figures")
    parser.add_argument("--no-save", action="store_true", help="Don't save results.json")
    args, unknown = parser.parse_known_args()

    main(num_tasks=args.tasks, output_dir=args.outdir, save_results=not args.no_save)

  HMP-DQN Scheduler Simulation
  Tasks: 1200  |  Seed: 42

[1/3] Training DQN agent...
  Step 400/1200  ε=0.1847  Avg EDP=1216.03 µJ·s
  Step 800/1200  ε=0.0500  Avg EDP=1211.44 µJ·s
  Step 1200/1200  ε=0.0500  Avg EDP=1145.86 µJ·s
      Final ε = 0.0500  |  Buffer = 1200
[2/3] Running Round-Robin baseline...
[3/3] Running Static (P-core) baseline...

  RESULTS SUMMARY
  EDP Reduction vs Round-Robin : +6.09%
  Exec Time Improvement        : +12.81%
  Energy Savings vs Static     : +15.17%
  DQN P-core usage             : 68.9% 
  Final ε (exploration)        : 0.05

✅  All figures saved to: figures/
  Results saved → results.json
